# 01 — Dataset Preparation

Sample the local Tiny-GenImage data, standardize images, and build the dataset manifest that every later notebook reads. See `data/README.md` and `configs/experiments.yaml` for more details.

In [ ]:
import sys
from pathlib import Path
from typing import TypedDict
import random
import pandas as pd
from PIL import Image

def find_repo_root(start: Path) -> Path:
    """Find the repository root from the current working directory."""
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "experiments.yaml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT))

from src.data.standardize import standardize_image, CorruptImageError
from src.data.config import load_config, active_sample_size, raw_data_root, raw_generator_dir

class SampledImage(TypedDict, total=False):
    """Metadata for an image sample used during dataset preparation.

    The record stores the original image path, class, source group,
    original and experimental splits, and standardized image path.
    """

    src_path: Path  # original file location under paths.raw_data_root
    label: str  # "ai" | "real"
    generator: str  # generator name for AI images, "real" for real images
    source_group: str  # generator folder from which the image was sampled
    raw_split: str  # original Tiny-GenImage split: "train" | "val"
    split: str  # experimental split: "train" | "val" | "test"
    dst_path: Path  # standardized JPEG location under data/processed/

## Config

Load the experiment settings from `configs/experiments.yaml`, including the random seed, selected generators, sample size, and preprocessing options.

In [ ]:
config: dict = load_config()
SEED: int = config["seed"]
GENERATORS: list[str] = config["generators"]
N_PER_GENERATOR: int = active_sample_size(config)
PREP: dict = config["preprocessing"]

print(f"seed = {SEED}")
print(f"generators: {GENERATORS}")
print(f"active size: {config['sample_size']['active_size']} -> {N_PER_GENERATOR} images per generator")
print(f"preprocessing: {PREP}")

## Locate the raw data

Set the path to the raw GenImage subset and check that the expected folder structure is available for each generator.

In [ ]:
# The raw GenImage data is large (~35k images) and stored outside the repository and loaded from the configured path.
RAW_ROOT: Path = raw_data_root(config)
assert RAW_ROOT.is_dir(), f"raw data root not found: {RAW_ROOT}"

print(f"Raw data root: {RAW_ROOT}\n")
for generator in GENERATORS:
    gen_dir: Path = raw_generator_dir(config, generator)
    # Only checking the "train" subfolder here as a quick existence probe
    ok: bool = gen_dir.is_dir() and (gen_dir / "train" / "ai").is_dir() and (gen_dir / "train" / "nature").is_dir()
    print(f"  {generator:<10} -> {gen_dir.name:<28} {'OK' if ok else 'MISSING'}")
    # Stop early if a generator is missing data.
    assert ok, f"missing/incomplete folder for generator {generator}: {gen_dir}"

## Sample images

Sample an equal number of AI and real images for each generator using the configured sample size and random seed. Train and validation examples come from the original Tiny-GenImage `train` folders, while test examples come exclusively from the original `val` folders. The original source group and split are retained for auditing.

In [ ]:
def list_raw_files(generator_dir: Path, raw_split: str, class_dir: str) -> list[Path]:
    """Return sorted image paths from one original dataset split.

    Args:
        generator_dir: Root directory for the generator.
        raw_split: Original Tiny-GenImage split, either "train" or "val".
        class_dir: Image class, either "ai" or "nature".

    Returns:
        Sorted image file paths from the requested directory.
    """
    directory = generator_dir / raw_split / class_dir
    return sorted(path for path in directory.iterdir() if path.is_file())

ratios: dict[str, float] = config["standard_split"]
n_train: int = int(N_PER_GENERATOR * ratios["train"])
n_val: int = int(N_PER_GENERATOR * ratios["val"])
n_test: int = N_PER_GENERATOR - n_train - n_val
rng = random.Random(SEED)

records: list[SampledImage] = []
for generator in GENERATORS:
    gen_dir = raw_generator_dir(config, generator)
    for label, class_dir in (("ai", "ai"), ("real", "nature")):
        raw_train_pool = list_raw_files(gen_dir, "train", class_dir)
        raw_val_pool = list_raw_files(gen_dir, "val", class_dir)
        assert len(raw_train_pool) >= n_train + n_val, (
            f"{generator}/{label}: need {n_train + n_val} raw train images, "
            f"found {len(raw_train_pool)}"
        )
        assert len(raw_val_pool) >= n_test, (
            f"{generator}/{label}: need {n_test} raw val images, "
            f"found {len(raw_val_pool)}"
        )

        train_val_sample = rng.sample(raw_train_pool, n_train + n_val)
        selected = (
            [(path, "train", "train") for path in train_val_sample[:n_train]]
            + [(path, "train", "val") for path in train_val_sample[n_train:]]
            + [(path, "val", "test") for path in rng.sample(raw_val_pool, n_test)]
        )

        records += [
            {
                "src_path": path,
                "label": label,
                "generator": generator if label == "ai" else "real",
                "source_group": generator,
                "raw_split": raw_split,
                "split": split,
            }
            for path, raw_split, split in selected
        ]

n_ai = sum(r["label"] == "ai" for r in records)
n_real = sum(r["label"] == "real" for r in records)
print(f"Sampled {len(records)} images total ({n_ai} ai, {n_real} real)")

## Standardize images

Apply the same preprocessing to all images: honor EXIF orientation, convert to RGB, center-crop to a square, resize to the configured resolution, remove metadata, and save with consistent JPEG compression (bias prevention).

In [ ]:
PROCESSED_ROOT: Path = REPO_ROOT / "data" / "processed"


def processed_path(record: SampledImage) -> Path:
    """Build the destination path for a standardized image.

    Args:
        record: Metadata for the sampled image.

    Returns:
        Path where the standardized JPEG image will be saved.
    """
    new_name = f"{record['source_group']}__{record['src_path'].stem}.jpg"
    return PROCESSED_ROOT / record["label"] / new_name


destination_paths: list[Path] = [processed_path(record) for record in records]
assert len(destination_paths) == len(set(destination_paths)), (
    "Multiple source images map to the same processed path."
)
for record, dst in zip(records, destination_paths):
    record["dst_path"] = dst

errors: list[str] = []
for i, record in enumerate(records):
    try:
        standardize_image(
            record["src_path"],
            record["dst_path"],
            image_size=PREP["image_size"],
            color_mode=PREP["color_mode"],
            jpeg_quality=PREP["jpeg_quality"],
        )
    except CorruptImageError as e:
        # Keep track of files that could not be standardized.
        errors.append(str(e))
    if (i + 1) % 2000 == 0:
        print(f"  standardized {i + 1}/{len(records)}")

print(f"\nDone. {len(records) - len(errors)} standardized, {len(errors)} failed.")
for e in errors[:20]:
    print(" ", e)

if errors:
    raise RuntimeError(
        f"Dataset preparation stopped because {len(errors)} images failed to standardize."
    )

## Build dataset

Create `data/dataset.csv` with columns: filename | label | generator | source_group | raw_split | split | source_path.

In [ ]:
# Build a portable manifest from the splits assigned during sampling.
rows: list[dict[str, str]] = [
    {
        "filename": r["dst_path"].relative_to(PROCESSED_ROOT).as_posix(),
        "label": r["label"],
        "generator": r["generator"],
        "source_group": r["source_group"],
        "raw_split": r["raw_split"],
        "split": r["split"],
        "source_path": r["src_path"].relative_to(RAW_ROOT).as_posix(),
    }
    for r in records
]

dataset: pd.DataFrame = pd.DataFrame(rows)
dataset_path: Path = REPO_ROOT / "data" / "dataset.csv"
dataset.to_csv(dataset_path, index=False)
print(f"Wrote {len(dataset)} rows to {dataset_path}")
dataset.head()

## Sanity checks

Check the dataset schema, class and source balance, standard and original split relationship, file paths, and standardized image properties.

In [ ]:
expected_columns = [
    "filename", "label", "generator", "source_group",
    "raw_split", "split", "source_path",
]
assert list(dataset.columns) == expected_columns, (
    f"unexpected dataset columns: {list(dataset.columns)}"
)

print("Split sizes:")
print(dataset["split"].value_counts(), "\n")

print("Label balance overall:")
print(dataset["label"].value_counts(), "\n")

print("AI counts per generator:")
print(dataset[dataset.label == "ai"].groupby("generator").size(), "\n")

print("Label balance per split:")
print(dataset.groupby(["split", "label"]).size().unstack(), "\n")

expected_split_counts = {"train": n_train, "val": n_val, "test": n_test}
for source_group in GENERATORS:
    for label in ("ai", "real"):
        group = dataset[(dataset["source_group"] == source_group) & (dataset["label"] == label)]
        assert group["split"].value_counts().to_dict() == expected_split_counts, (
            f"unexpected split counts for {source_group}/{label}: "
            f"{group['split'].value_counts().to_dict()}"
        )

assert (dataset.loc[dataset["label"] == "real", "generator"] == "real").all()
assert (dataset.loc[dataset["label"] == "ai", "generator"] == dataset.loc[dataset["label"] == "ai", "source_group"]).all()
assert (dataset.loc[dataset["split"] == "test", "raw_split"] == "val").all()
assert (dataset.loc[dataset["split"].isin(["train", "val"]), "raw_split"] == "train").all()

assert dataset["filename"].is_unique, "duplicate processed filenames in dataset!"
assert dataset["source_path"].is_unique, "duplicate raw source paths in dataset!"

missing_processed: list[str] = [
    filename for filename in dataset["filename"]
    if not (PROCESSED_ROOT / filename).is_file()
]
assert not missing_processed, (
    f"{len(missing_processed)} rows point at missing processed files, "
    f"e.g. {missing_processed[:5]}"
)

missing_raw: list[str] = [
    source_path for source_path in dataset["source_path"]
    if not (RAW_ROOT / source_path).is_file()
]
assert not missing_raw, (
    f"{len(missing_raw)} rows point at missing raw files, e.g. {missing_raw[:5]}"
)

# Check a reproducible sample for the expected encoding and image properties.
sample_filenames: list[str] = random.Random(SEED).sample(list(dataset["filename"]), 5)
for filename in sample_filenames:
    with Image.open(PROCESSED_ROOT / filename) as img:
        assert img.format == "JPEG", f"{filename}: bad format {img.format}"
        assert img.size == (PREP["image_size"], PREP["image_size"]), f"{filename}: bad size {img.size}"
        assert img.mode == PREP["color_mode"], f"{filename}: bad mode {img.mode}"
        assert len(img.getexif()) == 0, f"{filename}: EXIF metadata was not removed"

print("All sanity checks passed.")